# All sites plot Land Exposure compared to Low Tide


In [ ]:
import base64
import datetime
import html as html_module
import io
import os
import sys
import pathlib

import geopandas
import matplotlib.pyplot as plt
import numpy
import pandas
import plotly.express as px
import rasterio
import rasterio.features
import xarray
from IPython.display import FileLink, display

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import plotting

%load_ext autoreload
%autoreload 2

## Date-Range Site and Cloud Settings

Set the start and end years and cloud-cover cutoff below. All matching scenes are retained; low-tide timing is recorded for comparison, not used to exclude scenes.

In [ ]:
data_path = utils.get_data_path()

In [ ]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
sites = all_training_sites

years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

max_cloud_cover = 1

# Thumbnails for 1% cloud sorted by tide

In [ ]:
data_path = utils.get_data_path()

years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
sites = all_training_sites

max_cloud_cover = 1

_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=max_cloud_cover,
    predict_max_cloud_cover=max_cloud_cover,
    use_threshold=False, satellite_only=True) 

tide_time_file = satellite_path / "tide_time.csv"

In [ ]:
tide_times_all_sites = []

for site in sites:
    tide_time_series = []

    site_polygon_file = utils.get_site_polygon_path(site)
    site_polygon = geopandas.read_file(site_polygon_file)
    centroid_nzgd49 = site_polygon.to_crs(utils.CRS_NZGD49).geometry.union_all().centroid
    latitude_nzgd49, longitude_nzgd49 = float(centroid_nzgd49.y), float(centroid_nzgd49.x)
    
    for year in years:
        print(f"{site}: year {year}")
    
        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        if not satellite_file.exists():
            print(f"WARNING no {satellite_file.name}.")
            continue

        # load in satellite file
        satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
        # calculate distances from low tide
        timestamps = satellite_data["time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").values
        tide_info = []
        for timestamp in timestamps:
            tide_dict = sentinel2._get_low_tide_info_for_datetime(timestamp, latitude_nzgd49, longitude_nzgd49)
            tide_dict["time"] = timestamp
            tide_info.append(tide_dict)
        tide_time_series.append(pandas.DataFrame(tide_info))
    tide_time_series = pandas.concat(tide_time_series)
    tide_times_all_sites.append(tide_time_series)

# for plotting the gallery / distributions
tide_times_all_sites = pandas.concat(tide_times_all_sites, keys=sites, names=["site", "date id"])
tide_times_all_sites.to_csv(tide_time_file)

In [ ]:
tide_times_all_sites = pandas.read_csv(tide_time_file)
tide_times_all_sites["time"] = pandas.to_datetime(tide_times_all_sites["time"], utc=True)
tide_times_all_sites["minutes_from_low_tide"] = pandas.to_timedelta(tide_times_all_sites["time_from_low_tide"]).dt.total_seconds()/60
output_folder = satellite_path /  "rbg_thumbnails_tide_sorted"
output_folder.mkdir(exist_ok=True)

for site in sites:
    print(f"{site}")
    gallery_file = output_folder / f"{site}.html"
    if gallery_file.exists():
        continue
    plotting.create_rgb_gallery(site_name=site,
                                sort_value=tide_times_all_sites, 
                                sort_value_label="minutes_from_low_tide",
                                gallery_file=gallery_file, 
                                years=years,
                                max_cloud_cover=max_cloud_cover,
                                satellite_path=satellite_path)

In [ ]:
timestamps

In [ ]:
tide_info

# Thumbnails for 0% cloud band quality caluclations

In [ ]:
data_path = utils.get_data_path()

years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
sites = all_training_sites

baseline_max_cloud_cover = 0

_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=baseline_max_cloud_cover,
    predict_max_cloud_cover=baseline_max_cloud_cover,
    use_threshold=True, satellite_only=True) 

mean_mndwi_file = satellite_path / "mean_mndwi.csv"

In [ ]:
sites=sites[0:12]

In [ ]:
mean_mndwi_all_sites = []

for site in sites:
    mean_mndwi_series = []
    for year in years:
        print(f"{site}: year {year}")
    
        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        if not satellite_file.exists():
            print(f"WARNING no {satellite_file.name}.")
            continue

        # load in satellite file
        satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
        # calculate MNDWI layer
        mean_mndwi = sentinel2._mean_mndwi_by_date(satellite_data)
        mean_mndwi_series.append(
            mean_mndwi.compute().to_pandas()
        )
    mean_mndwi_series = pandas.concat(mean_mndwi_series)
    mean_mndwi_all_sites.append(mean_mndwi_series)

# for plotting the gallery / distributions
mean_mndwi_all_sites = pandas.concat(mean_mndwi_all_sites, keys=sites, names=["site", "time"]).rename("mean_mndwi")
mean_mndwi_all_sites.to_csv(mean_mndwi_file)

In [ ]:
mean_mndwi = pandas.read_csv(mean_mndwi_file)
output_folder = satellite_path /  "rbg_thumbnails_mndwi_sorted"
output_folder.mkdir(exist_ok=True)

for site in sites:
    print(f"{site}")
    gallery_file = output_folder / f"{site}.html"
    if gallery_file.exists():
        continue
    plotting.create_rgb_gallery(site_name=site,
                                sort_value=mean_mndwi, 
                                sort_value_label="Mean MNDWI",
                                gallery_file=gallery_file, 
                                years=years,
                                max_cloud_cover=baseline_max_cloud_cover,
                                satellite_path=satellite_path)

# Look at 0% cloud and review MNDWI

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()

In [ ]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

sites = all_training_sites
mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"
baseline_max_cloud_cover = 0
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [ ]:
_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=baseline_max_cloud_cover,
    predict_max_cloud_cover=baseline_max_cloud_cover,
    use_threshold=True, satellite_only=True) 

In [ ]:
mndwi_thresholds = pandas.read_csv(mndwi_file)

In [ ]:
mndwi_thresholds

In [ ]:
site = sites[0]
mndwi_threshold = mndwi_thresholds.loc[mndwi_thresholds["site"] == site, "threshold",].squeeze()

year = years[-1]
print(f"{site}: year {year}")

satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
site_polygon_file = utils.get_site_polygon_path(site)

if not satellite_file.exists():
    print(f"\tSave satellite image of the site {site} MNDWI thresholded and cloud filtered in the year {year}")
    site_polygon = geopandas.read_file(site_polygon_file)
    satellite_data = sentinel2.get_cloud_filtered_images_in_year(
        geometry=site_polygon,
        max_cloud_cover=baseline_max_cloud_cover,
        year=year,
    )

In [ ]:
satellite_data

In [ ]:
mndwi = sentinel2._mean_mndwi_by_date(satellite_data)

In [ ]:
mndwi

In [ ]:
green = satellite_data["B03"].astype(numpy.float32)
swir1 = satellite_data["B11"].astype(numpy.float32)
valid_pixels = (
    satellite_data["SCL"].notnull()
    & ~satellite_data["SCL"].isin(sentinel2.SCL_TO_IGNORE)
    & (green > 0)
    & (swir1 > 0)
)

((green - swir1) / (green + swir1)).where(valid_pixels).mean(dim=["x", "y"], skipna=True)

In [ ]:
satellite_data.time

In [ ]:
pandas.read_csv(data_path / "band_quality_baseline_tide_api.csv")

In [ ]:
pandas.read_csv(data_path / "band_quality_baseline_without_RGB_0_values.csv")

In [ ]:
_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=baseline_max_cloud_cover,
    predict_max_cloud_cover=baseline_max_cloud_cover,
    use_threshold=True, satellite_only=True) 
baseline = sentinel2.compute_and_save_band_quality_baseline(
    training_sites=sites[0:6],
    satellite_images_paths=[satellite_path],
)
baseline